# FloodShift 05B — Reliability & Modality-Dropout Hardening

Notebook 05 established a clear failure mode in the frozen 04H-CAL model:

- clean robustness-half mIoU: **0.8265**
- optical haze proxy mIoU: **0.5003**
- optical missing mIoU: **0.2917**
- SAR missing mIoU: **0.5609**
- combined mild mIoU: **0.4587**

Most importantly, optical-missing predictions became **more confident while becoming much less accurate**.

This notebook therefore performs **one predeclared hardening run**. It is not a new model search.

## Starting point

The starting segmenter is the exact frozen:

\[
\boxed{\text{04H epoch-4 weights} + b_{\mathrm{type}}=-0.15}
\]

Expected 04H SHA256:

```text
ef3341a7579f1316c80290dcd2a8f972c0adbf7e03e7a8c6bc239f30c4b47e53
```

## What is trainable

TerraMind and the decoder remain frozen.

Only these reliability-specific components are updated:

1. four multiscale `ReliabilityRouter` modules,
2. `DeepRIFT`,
3. the 04H hierarchical residual head.

This keeps the experiment tightly aligned with the failure diagnosed in Notebook 05.

## Training corruption distribution

Exactly one corruption state is sampled independently for each training item:

| state | probability |
|---|---:|
| clean | 0.45 |
| optical haze proxy | 0.15 |
| optical missing | 0.15 |
| SAR standardized-noise stress | 0.10 |
| SAR missing | 0.05 |
| combined mild degradation | 0.05 |
| reliability neutral | 0.05 |

The corruption **strengths are randomized inside fixed predeclared ranges** so that training does not simply memorize the exact Notebook-05 evaluation setting.

No reliability-inverted examples are used for training; that remains an adversarial evaluation condition.

## Fixed training budget

- **3 epochs exactly**
- one T4
- physical batch = 2
- gradient accumulation = 4
- effective batch = 8
- no hyperparameter sweep
- no epoch selection
- final epoch is evaluated

## Acceptance rule

05B is promoted only if it:

1. preserves clean full-validation quality,
2. materially improves the core missing/degraded-sensor robustness average,
3. prevents total temporary-flood collapse under optical loss,
4. does not materially damage SAR-missing robustness.

If it fails, the experiment is still scientifically useful and we fall back to 04H-CAL rather than launching an automatic 05C.

## Test policy

The 58-scene Pakistan + Spain frozen geographic test is **never wrapped in a Dataset or DataLoader** here.


In [ ]:
# ============================================================
# CELL: P13-C00 — KAGGLE-COMPAT TERRATORCH INSTALL
# ============================================================
# Proven compatible path from 04E.
# IMPORTANT: on a fresh Kaggle image, pip may change NumPy to 1.26.4.
# If the next cell reports a NumPy/Pandas ABI error, restart the Python
# kernel ONCE, do NOT rerun this cell, then continue from P10-C01.

import sys, subprocess

print("Installing TerraTorch 1.1.1 (04E/04F-proven compatibility path)...")
subprocess.check_call([
    sys.executable, "-m", "pip", "install",
    "-q", "--no-cache-dir", "terratorch==1.1.1"
])
print("P13-C00 COMPLETE ✅")
print("If NumPy changed, restart the Python kernel once before P10-C01.")

In [ ]:
# ============================================================
# CELL: P13-C01 — ENVIRONMENT + REPRODUCIBILITY
# ============================================================
import os
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")

import sys, gc, json, time, math, random, hashlib, zipfile
from pathlib import Path

import numpy as np
import scipy
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import rasterio
from rasterio.windows import Window

from sklearn.metrics import roc_auc_score, average_precision_score

import terratorch
from terratorch.registry import BACKBONE_REGISTRY

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
AMP_ENABLED = torch.cuda.is_available()
torch.backends.cudnn.benchmark = True

INPUT_ROOT = Path("/kaggle/input")
OUTPUT_ROOT = Path("/kaggle/working/FloodShift_05B_Reliability_Hardening")
CHECKPOINT_ROOT = OUTPUT_ROOT / "checkpoints"
METRIC_ROOT = OUTPUT_ROOT / "metrics"

for p in [OUTPUT_ROOT, CHECKPOINT_ROOT, METRIC_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

print("=" * 112)
print("P13-C01 — 05B ENVIRONMENT")
print("=" * 112)
print("Python     :", sys.version.split()[0])
print("NumPy      :", np.__version__)
print("Pandas     :", pd.__version__)
print("PyTorch    :", torch.__version__)
print("TerraTorch :", getattr(terratorch, "__version__", "unknown"))
print("Device     :", DEVICE)
if torch.cuda.is_available():
    print("GPU        :", torch.cuda.get_device_name(0))
print("AMP        :", AMP_ENABLED)
print("Output root:", OUTPUT_ROOT)


In [ ]:
# ============================================================
# CELL: P13-C02 — LOCATE + VERIFY FROZEN STARTING SYSTEM
# ============================================================
_file_cache = {}

def locate_by_saved_path(saved_path_or_name, required=True):
    saved = Path(str(saved_path_or_name))
    filename = saved.name
    key = str(saved_path_or_name)

    if key in _file_cache:
        return _file_cache[key]

    matches = []
    for root in [INPUT_ROOT]:
        if root.exists():
            matches.extend(root.rglob(filename))

    if not matches:
        if required:
            raise FileNotFoundError(f"Could not locate '{filename}' under /kaggle/input.")
        return None

    selected = sorted(matches, key=lambda p: (len(str(p)), str(p)))[0]
    _file_cache[key] = selected
    return selected

FINAL_MANIFEST_PATH = locate_by_saved_path("floodshift_preprocessing_manifest_v1.csv")
GLOBAL_LOCK_PATH = locate_by_saved_path("floodshift_preprocessing_GLOBAL_LOCK_v1.json")
OPTICAL_NORM_PATH = locate_by_saved_path("optical_10band_train_normalization_exact.csv")
SAR_SCENE_STATS_PATH = locate_by_saved_path("train_raw_sar_scene_statistics.csv")

CHECKPOINT_04H = locate_by_saved_path("FloodShift_04H_best_mIoU.pt")
CAL_CONFIG_PATH = locate_by_saved_path("FloodShift_04H_CAL_config.pt")

EXPECTED_04H_SHA256 = "ef3341a7579f1316c80290dcd2a8f972c0adbf7e03e7a8c6bc239f30c4b47e53"
EXPECTED_04E_SHA256 = "6ab8ad8b41aa64b4ceee403539d8554a14f61cab235a0f0841ef13f77e390822"
EXPECTED_TYPE_BIAS = -0.15

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

SHA_04H = sha256_file(CHECKPOINT_04H)
assert SHA_04H == EXPECTED_04H_SHA256, (
    f"Wrong 04H checkpoint: {SHA_04H}"
)

cal_cfg = torch.load(CAL_CONFIG_PATH, map_location="cpu", weights_only=False)
assert cal_cfg["source_04h_checkpoint_sha256"] == EXPECTED_04H_SHA256
assert cal_cfg["source_04e_sha256"] == EXPECTED_04E_SHA256
assert abs(float(cal_cfg["type_bias"]) - EXPECTED_TYPE_BIAS) < 1e-12
assert cal_cfg["decision"] == "04H_CAL_SINGLE_MODEL_PREFERRED"

TYPE_BIAS = float(cal_cfg["type_bias"])

with open(GLOBAL_LOCK_PATH, "r") as f:
    preprocessing_lock = json.load(f)

assert preprocessing_lock["status"] == "PERMANENTLY_FROZEN"
assert preprocessing_lock["preprocessing_development_closed"] is True

print("=" * 112)
print("P13-C02 — STARTING SYSTEM VERIFIED")
print("=" * 112)
print("04H checkpoint:", CHECKPOINT_04H)
print("04H SHA256    :", SHA_04H)
print("CAL config    :", CAL_CONFIG_PATH)
print("Type bias     :", TYPE_BIAS)
print("Preprocess    :", preprocessing_lock["status"])
print("P13-C02 COMPLETE ✅")


In [ ]:
# ============================================================
# CELL: P13-C03 — MANIFEST + FROZEN TRAIN-ONLY NORMALIZATION
# ============================================================

manifest = pd.read_csv(FINAL_MANIFEST_PATH)

assert len(manifest) == 440
assert manifest["scene_id"].nunique() == 440

train_df = (
    manifest[manifest["split"] == "train"]
    .copy()
    .reset_index(drop=True)
)

val_df = (
    manifest[manifest["split"] == "validation"]
    .copy()
    .reset_index(drop=True)
)

test_df = (
    manifest[manifest["split"] == "test"]
    .copy()
    .reset_index(drop=True)
)

assert len(train_df) == 314
assert len(val_df) == 68
assert len(test_df) == 58

optical_norm = pd.read_csv(OPTICAL_NORM_PATH)

OPTICAL_BANDS = optical_norm["band_name"].tolist()

EXPECTED_OPTICAL_BANDS = [
    "B2", "B3", "B4", "B5", "B6",
    "B7", "B8", "B8A", "B11", "B12"
]

assert OPTICAL_BANDS == EXPECTED_OPTICAL_BANDS

OPTICAL_MEAN = optical_norm["mean"].to_numpy(np.float32)
OPTICAL_STD = optical_norm["std"].to_numpy(np.float32)

# Original S2 source order:
# B1 B2 B3 B4 B5 B6 B7 B8 B8A B9 B10 B11 B12
OPTICAL_SOURCE_INDICES = np.array(
    [1, 2, 3, 4, 5, 6, 7, 8, 11, 12],
    dtype=np.int64,
)

sar_stats = pd.read_csv(SAR_SCENE_STATS_PATH)
assert len(sar_stats) == 314

n_all = sar_stats["valid_pixels"].to_numpy(np.float64)

def pooled_mean_std(mean_col, std_col):
    mu_all = sar_stats[mean_col].to_numpy(np.float64)
    sd_all = sar_stats[std_col].to_numpy(np.float64)

    valid = (
        np.isfinite(mu_all)
        & np.isfinite(sd_all)
        & (n_all > 0)
    )

    n = n_all[valid]
    mu = mu_all[valid]
    sd = sd_all[valid]

    total_n = n.sum()
    mean = np.sum(n * mu) / total_n
    second = np.sum(n * (sd**2 + mu**2)) / total_n
    var = max(second - mean**2, 0.0)

    return float(mean), float(np.sqrt(var))

VV_MEAN, VV_STD = pooled_mean_std(
    "vv_mean_db",
    "vv_std_db",
)

VH_MEAN, VH_STD = pooled_mean_std(
    "vh_mean_db",
    "vh_std_db",
)

SAR_MEAN = np.array(
    [VV_MEAN, VH_MEAN],
    dtype=np.float32,
)

SAR_STD = np.array(
    [VV_STD, VH_STD],
    dtype=np.float32,
)

assert np.isfinite(SAR_MEAN).all()
assert np.isfinite(SAR_STD).all()
assert (SAR_STD > 0).all()

print("=" * 112)
print("P13-C03 — DATA CONFIG")
print("=" * 112)
print("Train / Val / Test:", len(train_df), len(val_df), len(test_df))
print("Optical bands      :", OPTICAL_BANDS)
print("SAR mean [VV,VH]   :", SAR_MEAN)
print("SAR std  [VV,VH]   :", SAR_STD)
print("Reliability        : frozen continuous P(clear)")
print("Primary SAR input  : raw VV/VH")

In [ ]:
# ============================================================
# CELL: P13-C04 — TARGET + DATASET
#
# IMPORTANT:
# This intentionally preserves the CURRENT project's target mapping.
# It does NOT silently change label semantics mid-experiment.
# ============================================================

IGNORE_INDEX = 255
NUM_CLASSES = 3

CLASS_NAMES = [
    "non_flood",
    "permanent_water",
    "temporary_flood",
]

TILE_SIZE = 256
TILES_PER_SCENE = 4

TILE_OFFSETS = [
    (0, 0),
    (0, 256),
    (256, 0),
    (256, 256),
]


def resolve_scene_files(row):
    return {
        "sar": locate_by_saved_path(row["sar_raw_path"]),
        "optical": locate_by_saved_path(row["optical_raw_path"]),
        "flood": locate_by_saved_path(row["flood_label_path"]),
        "jrc": locate_by_saved_path(row["jrc_water_path"]),
        "reliability": locate_by_saved_path(
            row["optical_reliability_path"]
        ),
    }


def build_three_class_target(flood, jrc):
    flood = np.asarray(flood)
    jrc = np.asarray(jrc)

    target = np.full(
        flood.shape,
        IGNORE_INDEX,
        dtype=np.uint8,
    )

    valid = flood != -1

    target[valid] = 0

    # Current frozen project semantics:
    # any valid JRC permanent-water pixel -> class 1
    target[
        valid & (jrc == 1)
    ] = 1

    # flood-positive but not JRC permanent water -> class 2
    target[
        valid
        & (jrc != 1)
        & (flood == 1)
    ] = 2

    return target


def normalize_sar(sar):
    sar = sar.astype(
        np.float32,
        copy=False,
    )

    valid = np.isfinite(sar).all(axis=0)

    out = np.zeros_like(
        sar,
        dtype=np.float32,
    )

    for c in range(2):
        m = valid & np.isfinite(sar[c])

        out[c, m] = (
            sar[c, m] - SAR_MEAN[c]
        ) / SAR_STD[c]

    return out


def normalize_optical(optical_13):
    optical_13 = optical_13.astype(
        np.float32,
        copy=False,
    )

    valid = np.all(
        np.isfinite(optical_13)
        & (optical_13 != 0),
        axis=0,
    )

    reflectance = optical_13 / 10000.0

    optical_10 = reflectance[
        OPTICAL_SOURCE_INDICES
    ]

    out = (
        optical_10
        - OPTICAL_MEAN[:, None, None]
    ) / OPTICAL_STD[:, None, None]

    out = out.astype(
        np.float32,
        copy=False,
    )

    # Frozen Notebook-03 convention.
    out[:, ~valid] = 0.0

    return out


def augment_all(
    sar,
    optical,
    reliability,
    target,
):
    # D4-aligned augmentation only.
    if np.random.rand() < 0.5:
        sar = np.flip(sar, axis=2)
        optical = np.flip(optical, axis=2)
        reliability = np.flip(
            reliability,
            axis=1,
        )
        target = np.flip(
            target,
            axis=1,
        )

    if np.random.rand() < 0.5:
        sar = np.flip(sar, axis=1)
        optical = np.flip(optical, axis=1)
        reliability = np.flip(
            reliability,
            axis=0,
        )
        target = np.flip(
            target,
            axis=0,
        )

    k = np.random.randint(0, 4)

    if k:
        sar = np.rot90(
            sar,
            k=k,
            axes=(1, 2),
        )

        optical = np.rot90(
            optical,
            k=k,
            axes=(1, 2),
        )

        reliability = np.rot90(
            reliability,
            k=k,
            axes=(0, 1),
        )

        target = np.rot90(
            target,
            k=k,
            axes=(0, 1),
        )

    return (
        sar.copy(),
        optical.copy(),
        reliability.copy(),
        target.copy(),
    )


class FixedEvalDataset(Dataset):
    def __init__(
        self,
        dataframe,
        train_mode=False,
    ):
        self.df = dataframe.reset_index(
            drop=True
        )

        self.train_mode = train_mode

    def __len__(self):
        return (
            len(self.df)
            * TILES_PER_SCENE
        )

    def __getitem__(self, index):
        scene_index = (
            index // TILES_PER_SCENE
        )

        tile_index = (
            index % TILES_PER_SCENE
        )

        row = self.df.iloc[
            scene_index
        ]

        scene_id = str(
            row["scene_id"]
        )

        y0, x0 = TILE_OFFSETS[
            tile_index
        ]

        window = Window(
            x0,
            y0,
            TILE_SIZE,
            TILE_SIZE,
        )

        paths = resolve_scene_files(
            row
        )

        with rasterio.open(
            paths["sar"]
        ) as src:
            sar = src.read(
                window=window
            ).astype(np.float32)

        with rasterio.open(
            paths["optical"]
        ) as src:
            optical_13 = src.read(
                window=window
            ).astype(np.float32)

        with rasterio.open(
            paths["flood"]
        ) as src:
            flood = src.read(
                1,
                window=window,
            )

        with rasterio.open(
            paths["jrc"]
        ) as src:
            jrc = src.read(
                1,
                window=window,
            )

        reliability_full = np.load(
            paths["reliability"],
            mmap_mode="r",
        )

        reliability = np.asarray(
            reliability_full[
                y0:y0 + TILE_SIZE,
                x0:x0 + TILE_SIZE,
            ],
            dtype=np.float32,
        )

        sar = normalize_sar(sar)

        optical = normalize_optical(
            optical_13
        )

        target = build_three_class_target(
            flood,
            jrc,
        )

        reliability = np.clip(
            reliability,
            0.0,
            1.0,
        )

        if self.train_mode:
            (
                sar,
                optical,
                reliability,
                target,
            ) = augment_all(
                sar,
                optical,
                reliability,
                target,
            )

        return {
            "sar": torch.from_numpy(
                sar
            ),
            "optical": torch.from_numpy(
                optical
            ),
            "reliability": torch.from_numpy(
                reliability[
                    None,
                    ...,
                ]
            ),
            "target": torch.from_numpy(
                target.astype(
                    np.int64
                )
            ),
            "scene_id": scene_id,
            "tile_index": tile_index,
        }

In [ ]:
# ============================================================
# CELL: P13-C05 — TRAIN / VAL LOADERS + DETERMINISTIC VAL PARTITION
# ============================================================
TRAIN_BATCH_SIZE = 2
VAL_BATCH_SIZE = 2
GRAD_ACCUM_STEPS = 4
NUM_WORKERS = 2

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % (2**32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)

train_dataset = FixedEvalDataset(train_df, train_mode=True)

def stable_scene_hash(scene_id):
    token = ("FloodShift05|" + str(scene_id)).encode("utf-8")
    return hashlib.sha256(token).hexdigest()

val_partition = val_df[["scene_id"]].copy()
val_partition["partition_hash"] = val_partition["scene_id"].map(stable_scene_hash)
val_partition = val_partition.sort_values("partition_hash").reset_index(drop=True)

calibration_ids = set(val_partition.iloc[:34]["scene_id"].astype(str))
robustness_ids = set(val_partition.iloc[34:]["scene_id"].astype(str))

val_cal_df = val_df[val_df["scene_id"].astype(str).isin(calibration_ids)].copy().reset_index(drop=True)
val_robust_df = val_df[val_df["scene_id"].astype(str).isin(robustness_ids)].copy().reset_index(drop=True)

assert len(val_cal_df) == 34
assert len(val_robust_df) == 34
assert calibration_ids.isdisjoint(robustness_ids)

common = dict(
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    worker_init_fn=seed_worker,
    persistent_workers=(NUM_WORKERS > 0),
)

g = torch.Generator()
g.manual_seed(SEED)

train_loader = DataLoader(
    train_dataset,
    batch_size=TRAIN_BATCH_SIZE,
    shuffle=True,
    drop_last=False,
    generator=g,
    **common,
)

val_full_loader = DataLoader(
    FixedEvalDataset(val_df, train_mode=False),
    batch_size=VAL_BATCH_SIZE,
    shuffle=False,
    drop_last=False,
    **common,
)

val_cal_loader = DataLoader(
    FixedEvalDataset(val_cal_df, train_mode=False),
    batch_size=VAL_BATCH_SIZE,
    shuffle=False,
    drop_last=False,
    **common,
)

val_robust_loader = DataLoader(
    FixedEvalDataset(val_robust_df, train_mode=False),
    batch_size=VAL_BATCH_SIZE,
    shuffle=False,
    drop_last=False,
    **common,
)

batch = next(iter(val_full_loader))

print("=" * 112)
print("P13-C05 — LOADERS")
print("=" * 112)
print("Train scenes / tiles :", len(train_df), "/", len(train_dataset))
print("Full val scenes      :", len(val_df))
print("Calibration scenes   :", len(val_cal_df))
print("Robustness scenes    :", len(val_robust_df))
print("Frozen test scenes   :", len(test_df), "(LOCKED; no Dataset/DataLoader constructed)")
print("Effective train batch:", TRAIN_BATCH_SIZE * GRAD_ACCUM_STEPS)
assert len(test_df) == 58
print("P13-C05 COMPLETE ✅")


In [ ]:
# ============================================================
# CELL: P13-C06 — FINAL TERRAMIND-RIFT ARCHITECTURE
# ============================================================

# ------------------------------------------------------------
# TerraMind pretrained input conventions
# Official Sen1Floods11 config statistics.
# ------------------------------------------------------------

TM_S2_ALL_MEAN = np.array(
    [
        2357.089,
        2137.385,
        2018.788,
        2082.986,
        2295.651,
        2854.537,
        3122.849,
        3040.560,
        3306.481,
        1473.847,
        506.070,
        2472.825,
        1838.929,
    ],
    dtype=np.float32,
)

TM_S2_ALL_STD = np.array(
    [
        1624.683,
        1675.806,
        1557.708,
        1833.702,
        1823.738,
        1733.977,
        1732.131,
        1679.732,
        1727.260,
        1024.687,
        442.165,
        1331.411,
        1160.419,
    ],
    dtype=np.float32,
)

TM_S2_SUBSET_INDICES = np.array(
    [1, 2, 3, 4, 5, 6, 7, 8, 11, 12],
    dtype=np.int64,
)

TM_S2_MEAN = (
    TM_S2_ALL_MEAN[
        TM_S2_SUBSET_INDICES
    ]
)

TM_S2_STD = (
    TM_S2_ALL_STD[
        TM_S2_SUBSET_INDICES
    ]
)

TM_S1_MEAN = np.array(
    [-12.599, -20.293],
    dtype=np.float32,
)

TM_S1_STD = np.array(
    [5.195, 5.890],
    dtype=np.float32,
)

TM_OPTICAL_BAND_NAMES = [
    "BLUE",
    "GREEN",
    "RED",
    "RED_EDGE_1",
    "RED_EDGE_2",
    "RED_EDGE_3",
    "NIR_BROAD",
    "NIR_NARROW",
    "SWIR_1",
    "SWIR_2",
]

TM_LAYER_INDICES = [
    2,
    5,
    8,
    11,
]

TM_EMBED_DIM = 768
PYRAMID_CHANNELS = [
    64,
    128,
    256,
    512,
]

PYRAMID_SIZES = [
    128,
    64,
    32,
    16,
]


def make_group_norm(channels):
    for groups in [
        32,
        16,
        8,
        4,
        2,
        1,
    ]:
        if (
            channels
            % groups
            == 0
        ):
            return nn.GroupNorm(
                groups,
                channels,
            )

    raise RuntimeError(
        "No GroupNorm divisor."
    )


class ConvNormAct(nn.Module):
    def __init__(
        self,
        in_channels,
        out_channels,
        kernel_size=3,
    ):
        super().__init__()

        padding = (
            kernel_size // 2
        )

        self.block = nn.Sequential(
            nn.Conv2d(
                in_channels,
                out_channels,
                kernel_size=kernel_size,
                padding=padding,
                bias=False,
            ),
            make_group_norm(
                out_channels
            ),
            nn.GELU(),
        )

    def forward(self, x):
        return self.block(x)


class SEBlock(nn.Module):
    def __init__(
        self,
        channels,
        reduction=8,
    ):
        super().__init__()

        hidden = max(
            channels // reduction,
            8,
        )

        self.fc1 = nn.Conv2d(
            channels,
            hidden,
            kernel_size=1,
        )

        self.fc2 = nn.Conv2d(
            hidden,
            channels,
            kernel_size=1,
        )

    def forward(self, x):
        w = F.adaptive_avg_pool2d(
            x,
            1,
        )

        w = F.gelu(
            self.fc1(w)
        )

        w = torch.sigmoid(
            self.fc2(w)
        )

        return x * w


class ConvNeXtRefine(nn.Module):
    def __init__(
        self,
        channels,
        drop=0.0,
    ):
        super().__init__()

        self.dw = nn.Conv2d(
            channels,
            channels,
            kernel_size=7,
            padding=3,
            groups=channels,
        )

        self.norm = make_group_norm(
            channels
        )

        self.pw1 = nn.Conv2d(
            channels,
            channels * 4,
            kernel_size=1,
        )

        self.pw2 = nn.Conv2d(
            channels * 4,
            channels,
            kernel_size=1,
        )

        self.drop = nn.Dropout2d(
            drop
        )

        self.se = SEBlock(
            channels
        )

    def forward(self, x):
        residual = x

        x = self.dw(x)

        x = self.norm(x)

        x = F.gelu(
            self.pw1(x)
        )

        x = self.drop(
            self.pw2(x)
        )

        x = self.se(x)

        return residual + x


class ModalityProjector(nn.Module):
    def __init__(
        self,
        in_channels,
        out_channels,
        out_size,
    ):
        super().__init__()

        self.out_size = out_size

        self.proj = nn.Sequential(
            nn.Conv2d(
                in_channels,
                out_channels,
                kernel_size=1,
                bias=False,
            ),
            make_group_norm(
                out_channels
            ),
            nn.GELU(),
            ConvNeXtRefine(
                out_channels,
                drop=0.0,
            ),
        )

    def forward(self, x):
        x = self.proj(x)

        if (
            x.shape[-2:]
            != (
                self.out_size,
                self.out_size,
            )
        ):
            x = F.interpolate(
                x,
                size=(
                    self.out_size,
                    self.out_size,
                ),
                mode="bilinear",
                align_corners=False,
            )

        return x


class ReliabilityRouter(nn.Module):
    """
    Spatial SAR-optical router.

    P(clear) is used as a prior on the optical route,
    while the learned router can still override that prior.
    """

    def __init__(
        self,
        channels,
        hidden=64,
        prior_strength=0.50,
    ):
        super().__init__()

        hidden = min(
            hidden,
            max(
                channels // 2,
                16,
            ),
        )

        self.prior_strength = (
            float(
                prior_strength
            )
        )

        self.net = nn.Sequential(
            nn.Conv2d(
                channels * 2 + 1,
                hidden,
                kernel_size=3,
                padding=1,
                bias=False,
            ),
            make_group_norm(
                hidden
            ),
            nn.GELU(),
            nn.Conv2d(
                hidden,
                2,
                kernel_size=1,
            ),
        )

        # Start near neutral learned routing.
        nn.init.zeros_(
            self.net[-1].weight
        )

        nn.init.zeros_(
            self.net[-1].bias
        )

    def forward(
        self,
        sar,
        optical,
        pclear,
    ):
        r = F.interpolate(
            pclear,
            size=sar.shape[-2:],
            mode="bilinear",
            align_corners=False,
        )

        r = r.clamp(
            0.0,
            1.0,
        )

        logits = self.net(
            torch.cat(
                [
                    sar,
                    optical,
                    r,
                ],
                dim=1,
            )
        )

        # Optical prior. Clamp avoids -inf at P(clear)=0.
        logits[
            :,
            1:2,
        ] = (
            logits[
                :,
                1:2,
            ]
            + self.prior_strength
            * torch.log(
                r.clamp(
                    min=0.05,
                    max=1.0,
                )
            )
        )

        weights = torch.softmax(
            logits,
            dim=1,
        )

        w_sar = weights[
            :,
            0:1,
        ]

        w_opt = weights[
            :,
            1:2,
        ]

        fused = (
            w_sar * sar
            + w_opt * optical
        )

        return (
            fused,
            weights,
        )


class DeepRIFT(nn.Module):
    """
    Zero-initialized reliability-conditioned cross attention.

    When P(clear) is high:
      SAR can borrow more optical context.

    When P(clear) is low:
      optical can borrow more SAR context.

    Output projections start at zero, so the block is an
    exact identity at initialization.
    """

    def __init__(
        self,
        channels=512,
        heads=8,
        dropout=0.0,
    ):
        super().__init__()

        self.channels = channels

        self.sar_norm = nn.LayerNorm(
            channels
        )

        self.opt_norm = nn.LayerNorm(
            channels
        )

        self.sar_from_opt = (
            nn.MultiheadAttention(
                channels,
                num_heads=heads,
                dropout=dropout,
                batch_first=True,
            )
        )

        self.opt_from_sar = (
            nn.MultiheadAttention(
                channels,
                num_heads=heads,
                dropout=dropout,
                batch_first=True,
            )
        )

        self.sar_out = nn.Linear(
            channels,
            channels,
        )

        self.opt_out = nn.Linear(
            channels,
            channels,
        )

        nn.init.zeros_(
            self.sar_out.weight
        )

        nn.init.zeros_(
            self.sar_out.bias
        )

        nn.init.zeros_(
            self.opt_out.weight
        )

        nn.init.zeros_(
            self.opt_out.bias
        )

    @staticmethod
    def _flatten(x):
        return (
            x.flatten(2)
            .transpose(1, 2)
            .contiguous()
        )

    @staticmethod
    def _unflatten(
        tokens,
        height,
        width,
    ):
        return (
            tokens
            .transpose(1, 2)
            .reshape(
                tokens.shape[0],
                tokens.shape[2],
                height,
                width,
            )
            .contiguous()
        )

    def forward(
        self,
        sar,
        optical,
        pclear,
    ):
        b, c, h, w = sar.shape

        sar_t = self._flatten(
            sar
        )

        opt_t = self._flatten(
            optical
        )

        r = F.adaptive_avg_pool2d(
            pclear,
            output_size=(h, w),
        )

        r_t = (
            r.flatten(2)
            .transpose(1, 2)
            .clamp(0.0, 1.0)
        )

        sar_n = self.sar_norm(
            sar_t
        )

        opt_n = self.opt_norm(
            opt_t
        )

        # Clear optical context receives larger K/V magnitude.
        opt_context = (
            opt_n
            * (
                0.25
                + 0.75 * r_t
            )
        )

        sar_delta, _ = (
            self.sar_from_opt(
                query=sar_n,
                key=opt_context,
                value=opt_context,
                need_weights=False,
            )
        )

        # Under low P(clear), the optical branch borrows SAR more.
        opt_delta, _ = (
            self.opt_from_sar(
                query=opt_n,
                key=sar_n,
                value=sar_n,
                need_weights=False,
            )
        )

        sar_t = (
            sar_t
            + r_t
            * self.sar_out(
                sar_delta
            )
        )

        opt_t = (
            opt_t
            + (1.0 - r_t)
            * self.opt_out(
                opt_delta
            )
        )

        sar = self._unflatten(
            sar_t,
            h,
            w,
        )

        optical = self._unflatten(
            opt_t,
            h,
            w,
        )

        return sar, optical


class DecoderBlock(nn.Module):
    def __init__(
        self,
        in_channels,
        skip_channels,
        out_channels,
    ):
        super().__init__()

        self.fuse = nn.Sequential(
            ConvNormAct(
                in_channels
                + skip_channels,
                out_channels,
                kernel_size=3,
            ),
            ConvNeXtRefine(
                out_channels,
                drop=0.05,
            ),
            ConvNeXtRefine(
                out_channels,
                drop=0.05,
            ),
        )

    def forward(
        self,
        x,
        skip,
    ):
        x = F.interpolate(
            x,
            size=skip.shape[-2:],
            mode="bilinear",
            align_corners=False,
        )

        x = torch.cat(
            [
                x,
                skip,
            ],
            dim=1,
        )

        return self.fuse(x)


class FloodShiftTerraMindRIFT(nn.Module):
    def __init__(
        self,
        num_classes=3,
        pretrained_backbone=False,
    ):
        super().__init__()

        # concat is used deliberately:
        # each spatial token contains S2 and S1 embeddings
        # concatenated in the modality order supplied below.
        self.backbone = (
            BACKBONE_REGISTRY.build(
                "terramind_v1_base",
                pretrained=pretrained_backbone,
                img_size=TILE_SIZE,
                modalities=[
                    "S2L1C",
                    "S1GRD",
                ],
                bands={
                    "S2L1C":
                        TM_OPTICAL_BAND_NAMES,
                },
                merge_method="concat",
            )
        )

        self.register_buffer(
            "project_opt_mean",
            torch.tensor(
                OPTICAL_MEAN,
                dtype=torch.float32,
            ).view(
                1,
                10,
                1,
                1,
            ),
            persistent=True,
        )

        self.register_buffer(
            "project_opt_std",
            torch.tensor(
                OPTICAL_STD,
                dtype=torch.float32,
            ).view(
                1,
                10,
                1,
                1,
            ),
            persistent=True,
        )

        self.register_buffer(
            "project_sar_mean",
            torch.tensor(
                SAR_MEAN,
                dtype=torch.float32,
            ).view(
                1,
                2,
                1,
                1,
            ),
            persistent=True,
        )

        self.register_buffer(
            "project_sar_std",
            torch.tensor(
                SAR_STD,
                dtype=torch.float32,
            ).view(
                1,
                2,
                1,
                1,
            ),
            persistent=True,
        )

        self.register_buffer(
            "tm_opt_mean",
            torch.tensor(
                TM_S2_MEAN,
                dtype=torch.float32,
            ).view(
                1,
                10,
                1,
                1,
            ),
            persistent=True,
        )

        self.register_buffer(
            "tm_opt_std",
            torch.tensor(
                TM_S2_STD,
                dtype=torch.float32,
            ).view(
                1,
                10,
                1,
                1,
            ),
            persistent=True,
        )

        self.register_buffer(
            "tm_sar_mean",
            torch.tensor(
                TM_S1_MEAN,
                dtype=torch.float32,
            ).view(
                1,
                2,
                1,
                1,
            ),
            persistent=True,
        )

        self.register_buffer(
            "tm_sar_std",
            torch.tensor(
                TM_S1_STD,
                dtype=torch.float32,
            ).view(
                1,
                2,
                1,
                1,
            ),
            persistent=True,
        )

        self.sar_projectors = (
            nn.ModuleList()
        )

        self.opt_projectors = (
            nn.ModuleList()
        )

        self.routers = nn.ModuleList()

        for (
            channels,
            out_size,
        ) in zip(
            PYRAMID_CHANNELS,
            PYRAMID_SIZES,
        ):
            self.sar_projectors.append(
                ModalityProjector(
                    TM_EMBED_DIM,
                    channels,
                    out_size,
                )
            )

            self.opt_projectors.append(
                ModalityProjector(
                    TM_EMBED_DIM,
                    channels,
                    out_size,
                )
            )

            self.routers.append(
                ReliabilityRouter(
                    channels,
                    prior_strength=0.50,
                )
            )

        self.deep_rift = DeepRIFT(
            channels=PYRAMID_CHANNELS[-1],
            heads=8,
            dropout=0.0,
        )

        self.deep_context = nn.Sequential(
            ConvNeXtRefine(
                PYRAMID_CHANNELS[-1],
                drop=0.05,
            ),
            ConvNeXtRefine(
                PYRAMID_CHANNELS[-1],
                drop=0.05,
            ),
        )

        self.decoder3 = DecoderBlock(
            512,
            256,
            256,
        )

        self.decoder2 = DecoderBlock(
            256,
            128,
            128,
        )

        self.decoder1 = DecoderBlock(
            128,
            64,
            64,
        )

        self.final_refine = nn.Sequential(
            ConvNormAct(
                64,
                64,
                kernel_size=3,
            ),
            ConvNeXtRefine(
                64,
                drop=0.05,
            ),
        )

        self.segmentation_head = (
            nn.Conv2d(
                64,
                num_classes,
                kernel_size=1,
            )
        )

        self.aux_head = nn.Sequential(
            nn.Conv2d(
                128,
                64,
                kernel_size=3,
                padding=1,
                bias=False,
            ),
            make_group_norm(64),
            nn.GELU(),
            nn.Conv2d(
                64,
                num_classes,
                kernel_size=1,
            ),
        )

    def to_terramind_inputs(
        self,
        sar_z,
        optical_z,
    ):
        # Detect frozen invalid-fill pixels BEFORE inverse affine.
        opt_invalid = (
            optical_z
            .abs()
            .sum(
                dim=1,
                keepdim=True,
            )
            < 1e-12
        )

        sar_invalid = (
            sar_z
            .abs()
            .sum(
                dim=1,
                keepdim=True,
            )
            < 1e-12
        )

        # Frozen optical z-score -> reflectance -> DN.
        optical_reflectance = (
            optical_z
            * self.project_opt_std
            + self.project_opt_mean
        )

        optical_dn = (
            optical_reflectance
            * 10000.0
        )

        optical_tm = (
            optical_dn
            - self.tm_opt_mean
        ) / self.tm_opt_std

        # Frozen SAR z-score -> raw dB.
        sar_db = (
            sar_z
            * self.project_sar_std
            + self.project_sar_mean
        )

        sar_tm = (
            sar_db
            - self.tm_sar_mean
        ) / self.tm_sar_std

        # Preserve invalid-fill convention in model space.
        optical_tm = optical_tm.masked_fill(
            opt_invalid,
            0.0,
        )

        sar_tm = sar_tm.masked_fill(
            sar_invalid,
            0.0,
        )

        return (
            sar_tm,
            optical_tm,
        )

    @staticmethod
    def tokens_to_map(tokens):
        b, n, c = tokens.shape

        side = int(
            math.isqrt(n)
        )

        if (
            side * side
            != n
        ):
            raise RuntimeError(
                f"TerraMind token count {n} "
                "is not a square grid."
            )

        return (
            tokens
            .transpose(1, 2)
            .reshape(
                b,
                c,
                side,
                side,
            )
            .contiguous()
        )

    def extract_modality_tokens(
        self,
        tm_outputs,
    ):
        if (
            not isinstance(
                tm_outputs,
                (list, tuple),
            )
            or len(tm_outputs) < 12
        ):
            raise RuntimeError(
                "Unexpected TerraMind output. "
                "Expected list of 12 transformer-layer outputs."
            )

        sar_tokens = []
        opt_tokens = []

        for layer_index in TM_LAYER_INDICES:
            tokens = tm_outputs[
                layer_index
            ]

            if (
                not torch.is_tensor(tokens)
                or tokens.ndim != 3
            ):
                raise RuntimeError(
                    "TerraMind concat output should be "
                    "[B, tokens, embedding]."
                )

            expected_dim = (
                TM_EMBED_DIM * 2
            )

            if (
                tokens.shape[-1]
                != expected_dim
            ):
                raise RuntimeError(
                    "Expected concat embedding dimension "
                    f"{expected_dim}, got "
                    f"{tokens.shape[-1]}. "
                    "Check TerraTorch version and merge_method."
                )

            # modalities were supplied in order:
            # [S2L1C, S1GRD]
            optical_part = tokens[
                :,
                :,
                :TM_EMBED_DIM,
            ]

            sar_part = tokens[
                :,
                :,
                TM_EMBED_DIM:,
            ]

            opt_tokens.append(
                optical_part
            )

            sar_tokens.append(
                sar_part
            )

        return (
            sar_tokens,
            opt_tokens,
        )

    def forward(
        self,
        sar,
        optical,
        reliability,
        return_aux=False,
    ):
        (
            sar_tm,
            optical_tm,
        ) = self.to_terramind_inputs(
            sar,
            optical,
        )

        tm_outputs = self.backbone(
            {
                "S2L1C": optical_tm,
                "S1GRD": sar_tm,
            }
        )

        (
            sar_tokens,
            opt_tokens,
        ) = self.extract_modality_tokens(
            tm_outputs
        )

        sar_maps = []
        opt_maps = []

        for i in range(4):
            sar_map = (
                self.tokens_to_map(
                    sar_tokens[i]
                )
            )

            opt_map = (
                self.tokens_to_map(
                    opt_tokens[i]
                )
            )

            sar_map = (
                self.sar_projectors[i](
                    sar_map
                )
            )

            opt_map = (
                self.opt_projectors[i](
                    opt_map
                )
            )

            sar_maps.append(
                sar_map
            )

            opt_maps.append(
                opt_map
            )

        # RIFT only at deepest 16x16 level.
        (
            sar_maps[-1],
            opt_maps[-1],
        ) = self.deep_rift(
            sar_maps[-1],
            opt_maps[-1],
            reliability,
        )

        fused = []
        router_weights = []

        for i in range(4):
            (
                fused_i,
                weights_i,
            ) = self.routers[i](
                sar_maps[i],
                opt_maps[i],
                reliability,
            )

            fused.append(
                fused_i
            )

            router_weights.append(
                weights_i
            )

        f0, f1, f2, f3 = fused

        x = self.deep_context(
            f3
        )

        x = self.decoder3(
            x,
            f2,
        )

        x = self.decoder2(
            x,
            f1,
        )

        aux_logits = self.aux_head(
            x
        )

        x = self.decoder1(
            x,
            f0,
        )

        x = F.interpolate(
            x,
            size=(
                TILE_SIZE,
                TILE_SIZE,
            ),
            mode="bilinear",
            align_corners=False,
        )

        x = self.final_refine(
            x
        )

        logits = (
            self.segmentation_head(
                x
            )
        )

        if return_aux:
            return {
                "logits": logits,
                "aux_logits": aux_logits,
                "router_weights": router_weights,
            }

        return logits

In [ ]:
# ============================================================
# CELL: P13-C07 — REBUILD 04H-CAL + SELECTIVE RELIABILITY UNFREEZE
# ============================================================
class HierarchicalResidualHead(nn.Module):
    def __init__(self, in_channels=16, hidden=48, max_delta=1.50):
        super().__init__()
        self.max_delta = float(max_delta)
        self.net = nn.Sequential(
            ConvNormAct(in_channels, hidden, 3),
            ConvNeXtRefine(hidden, drop=0.05),
            ConvNormAct(hidden, hidden, 3),
            nn.Conv2d(hidden, 2, kernel_size=1),
        )

    def forward(self, base_prob, sar, optical, reliability):
        x = torch.cat([base_prob, sar, optical, reliability], dim=1)
        raw = self.net(x).float()
        return self.max_delta * torch.tanh(raw / self.max_delta)

class FloodShift05B(nn.Module):
    """
    04H-CAL with a fixed type bias.
    During 05B only ReliabilityRouter + DeepRIFT + 04H head are trainable.
    """
    def __init__(self, base, type_bias):
        super().__init__()
        self.base = base
        self.head = HierarchicalResidualHead()
        self.type_bias = float(type_bias)

    @property
    def backbone(self):
        return self.base.backbone

    def train(self, mode=True):
        super().train(mode)

        # Frozen network stays in eval mode.
        self.base.eval()

        # Explicitly train only the reliability-specific modules.
        self.base.routers.train(mode)
        self.base.deep_rift.train(mode)
        self.head.train(mode)
        return self

    def forward(self, sar, optical, reliability, return_aux=False):
        # No blanket no_grad here: selected base modules need gradients.
        # The frozen backbone/projectors/decoder have requires_grad=False,
        # so autograd begins only at trainable reliability modules.
        base_out = self.base(
            sar,
            optical,
            reliability,
            return_aux=True,
        )

        base_logits = base_out["logits"].float()
        base_prob = torch.softmax(base_logits, dim=1)

        correction = self.head(
            base_prob.to(sar.dtype),
            sar,
            optical,
            reliability,
        )

        delta_water = correction[:, 0]
        delta_type = correction[:, 1]

        base_water_logit = (
            torch.logsumexp(base_logits[:, 1:3], dim=1)
            - base_logits[:, 0]
        )
        base_type_logit = base_logits[:, 2] - base_logits[:, 1]

        water_logit = base_water_logit + delta_water
        type_logit = base_type_logit + delta_type + self.type_bias

        p_water = torch.sigmoid(water_logit)
        p_temp_given_water = torch.sigmoid(type_logit)

        p0 = 1.0 - p_water
        p1 = p_water * (1.0 - p_temp_given_water)
        p2 = p_water * p_temp_given_water

        probs = torch.stack([p0, p1, p2], dim=1).clamp_min(1e-7)
        log_probs = probs.log()

        if return_aux:
            return {
                "logits": log_probs,
                "probs": probs,
                "base_logits": base_logits,
                "base_prob": base_prob,
                "water_logit": water_logit,
                "type_logit": type_logit,
                "delta_water": delta_water,
                "delta_type": delta_type,
                "router_weights": base_out["router_weights"],
            }

        return log_probs

payload_04h = torch.load(CHECKPOINT_04H, map_location="cpu", weights_only=False)

assert payload_04h.get("base_sha256") == EXPECTED_04E_SHA256
assert int(payload_04h.get("epoch", -1)) == 4
assert payload_04h.get("tag") == "best_mIoU"

base = FloodShiftTerraMindRIFT(num_classes=NUM_CLASSES, pretrained_backbone=False)
model = FloodShift05B(base, TYPE_BIAS)

missing, unexpected = model.load_state_dict(payload_04h["state_dict"], strict=False)
assert not missing, f"Missing keys: {missing[:10]}"
assert not unexpected, f"Unexpected keys: {unexpected[:10]}"

# Freeze absolutely everything first.
for p in model.parameters():
    p.requires_grad = False

# Then selectively unfreeze reliability-specific modules.
for p in model.base.routers.parameters():
    p.requires_grad = True

for p in model.base.deep_rift.parameters():
    p.requires_grad = True

for p in model.head.parameters():
    p.requires_grad = True

model = model.to(DEVICE).eval()

trainable_names = [
    n for n, p in model.named_parameters()
    if p.requires_grad
]

assert trainable_names
assert all(
    n.startswith("base.routers.")
    or n.startswith("base.deep_rift.")
    or n.startswith("head.")
    for n in trainable_names
)

print("=" * 112)
print("P13-C07 — SELECTIVE HARDENING MODEL")
print("=" * 112)
print("Type bias            :", TYPE_BIAS, "(fixed)")
print("Total params         :", f"{sum(p.numel() for p in model.parameters()):,}")
print("Trainable params     :", f"{sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
print("Trainable tensor count:", len(trainable_names))
print("Trainable groups     : routers + DeepRIFT + 04H head")
print("TerraMind backbone   : FROZEN")
print("Projectors/decoder   : FROZEN")
print("P13-C07 COMPLETE ✅")


In [ ]:
# ============================================================
# CELL: P13-C08 — METRICS + EXACT 4-WAY TTA
# ============================================================
TTA_MODES = ["identity", "hflip", "vflip", "rot180"]

def confusion_from_predictions(pred, target):
    valid = target != IGNORE_INDEX
    target_v = target[valid].long()
    pred_v = pred[valid].long()

    indices = NUM_CLASSES * target_v + pred_v
    return torch.bincount(
        indices,
        minlength=NUM_CLASSES * NUM_CLASSES,
    ).reshape(NUM_CLASSES, NUM_CLASSES)

def metrics_from_confusion(cm):
    if torch.is_tensor(cm):
        cm = cm.detach().cpu().numpy().astype(np.float64)
    else:
        cm = np.asarray(cm, dtype=np.float64)

    tp = np.diag(cm)
    fp = cm.sum(axis=0) - tp
    fn = cm.sum(axis=1) - tp

    den_iou = tp + fp + fn
    iou = np.divide(tp, den_iou, out=np.full_like(tp, np.nan), where=den_iou > 0)

    den_dice = 2 * tp + fp + fn
    dice = np.divide(2 * tp, den_dice, out=np.full_like(tp, np.nan), where=den_dice > 0)

    accuracy = float(tp.sum() / max(cm.sum(), 1.0))

    out = {
        "mIoU": float(np.nanmean(iou)),
        "macro_dice": float(np.nanmean(dice)),
        "accuracy": accuracy,
        "confusion": cm.tolist(),
    }

    for i, name in enumerate(CLASS_NAMES):
        out[f"{name}_IoU"] = float(iou[i])
        out[f"{name}_Dice"] = float(dice[i])

    return out

def compact_metrics(m):
    return {
        "mIoU": round(m["mIoU"], 4),
        "non_flood": round(m["non_flood_IoU"], 4),
        "permanent": round(m["permanent_water_IoU"], 4),
        "temporary": round(m["temporary_flood_IoU"], 4),
    }

def _tta_transform(tensor, mode):
    if mode == "identity":
        return tensor
    if mode == "hflip":
        return torch.flip(tensor, dims=[-1])
    if mode == "vflip":
        return torch.flip(tensor, dims=[-2])
    if mode == "rot180":
        return torch.rot90(tensor, k=2, dims=(-2, -1))
    raise ValueError(mode)

def _tta_inverse(tensor, mode):
    return _tta_transform(tensor, mode)

@torch.inference_mode()
def tta_log_probs(model_to_eval, sar, optical, reliability):
    total = None
    model_to_eval.eval()

    for mode in TTA_MODES:
        sar_t = _tta_transform(sar, mode)
        opt_t = _tta_transform(optical, mode)
        rel_t = _tta_transform(reliability, mode)

        with torch.amp.autocast("cuda", enabled=AMP_ENABLED):
            lp = model_to_eval(sar_t, opt_t, rel_t, return_aux=False)

        lp = _tta_inverse(lp, mode).float()
        total = lp if total is None else total + lp

    return total / len(TTA_MODES)

@torch.inference_mode()
def evaluate_4way(model_to_eval, loader):
    cm = torch.zeros((NUM_CLASSES, NUM_CLASSES), dtype=torch.long)
    model_to_eval.eval()

    for bi, b in enumerate(loader, start=1):
        sar = b["sar"].to(DEVICE, non_blocking=True)
        opt = b["optical"].to(DEVICE, non_blocking=True)
        rel = b["reliability"].to(DEVICE, non_blocking=True)
        y = b["target"].to(DEVICE, non_blocking=True)

        lp = tta_log_probs(model_to_eval, sar, opt, rel)
        pred = lp.argmax(1)
        cm += confusion_from_predictions(pred, y).cpu()

        if bi == 1 or bi % 30 == 0 or bi == len(loader):
            print(f"[{bi:03d}/{len(loader)}] validation")

    return metrics_from_confusion(cm)

print("P13-C08 COMPLETE ✅")


In [ ]:
# ============================================================
# CELL: P13-C09 — MANDATORY WARM-START 04H-CAL REPRODUCTION
# ============================================================
EXPECTED_04H_CAL = {
    "mIoU": 0.796029,
    "non_flood_IoU": 0.976510,
    "permanent_water_IoU": 0.825058,
    "temporary_flood_IoU": 0.586519,
}

warm_metrics = evaluate_4way(model, val_full_loader)

print("=" * 112)
print("P13-C09 — WARM START")
print("=" * 112)
print("Reproduced:", compact_metrics(warm_metrics))
print("Expected  :", {
    "mIoU": 0.7960,
    "non_flood": 0.9765,
    "permanent": 0.8251,
    "temporary": 0.5865,
})

for k, ref in EXPECTED_04H_CAL.items():
    assert abs(warm_metrics[k] - ref) < 0.0015, (
        f"Warm-start reproduction failed for {k}: {warm_metrics[k]} vs {ref}"
    )

print("P13-C09 COMPLETE ✅")


In [ ]:
# ============================================================
# CELL: P13-C10 — PREDECLARED CORRUPTION CURRICULUM
# ============================================================
TRAIN_SCENARIOS = [
    "clean",
    "optical_haze",
    "optical_missing",
    "sar_noise",
    "sar_missing",
    "combined_mild",
    "reliability_neutral",
]

TRAIN_SCENARIO_PROBS = np.array(
    [0.45, 0.15, 0.15, 0.10, 0.05, 0.05, 0.05],
    dtype=np.float64,
)
assert abs(TRAIN_SCENARIO_PROBS.sum() - 1.0) < 1e-12

SCENARIO_TO_CODE = {name: i for i, name in enumerate(TRAIN_SCENARIOS)}
CODE_TO_SCENARIO = {i: name for name, i in SCENARIO_TO_CODE.items()}

train_corruption_rng = np.random.default_rng(SEED + 505)

def _noise_like_single(x, seed):
    g = torch.Generator(device=x.device)
    g.manual_seed(int(seed))
    return torch.randn(
        x.shape,
        device=x.device,
        dtype=x.dtype,
        generator=g,
    )

def corrupt_training_batch(sar, optical, reliability, epoch, batch_index):
    sar_c = sar.clone()
    opt_c = optical.clone()
    rel_c = reliability.clone()

    bsz = sar.shape[0]

    names = train_corruption_rng.choice(
        TRAIN_SCENARIOS,
        size=bsz,
        p=TRAIN_SCENARIO_PROBS,
    )

    codes = torch.tensor(
        [SCENARIO_TO_CODE[n] for n in names],
        dtype=torch.long,
        device=sar.device,
    )

    for i, name in enumerate(names):
        if name == "clean":
            continue

        if name in ("optical_haze", "combined_mild"):
            kernel = int(train_corruption_rng.choice([3, 5, 7]))
            contrast = float(train_corruption_rng.uniform(0.55, 0.80))
            rel_scale = float(train_corruption_rng.uniform(0.15, 0.50))

            opt_i = opt_c[i:i+1]
            opt_c[i:i+1] = contrast * F.avg_pool2d(
                opt_i,
                kernel_size=kernel,
                stride=1,
                padding=kernel // 2,
            )
            rel_c[i:i+1] *= rel_scale

        if name == "optical_missing":
            opt_c[i].zero_()
            rel_c[i].zero_()

        if name in ("sar_noise", "combined_mild"):
            if name == "sar_noise":
                sigma = float(train_corruption_rng.uniform(0.15, 0.35))
            else:
                sigma = float(train_corruption_rng.uniform(0.15, 0.25))

            seed = (
                SEED * 1_000_000
                + int(epoch) * 100_000
                + int(batch_index) * 100
                + int(i)
            )

            noise = _noise_like_single(sar_c[i:i+1], seed)
            valid = (
                sar_c[i:i+1].abs().sum(dim=1, keepdim=True) > 0
            ).to(sar_c.dtype)

            sar_c[i:i+1] = sar_c[i:i+1] + sigma * noise * valid

        if name == "sar_missing":
            sar_c[i].zero_()

        if name == "reliability_neutral":
            rel_c[i].fill_(0.5)

    return sar_c, opt_c, rel_c, codes, list(names)

print("=" * 112)
print("P13-C10 — TRAINING CORRUPTION DISTRIBUTION")
print("=" * 112)
for name, p in zip(TRAIN_SCENARIOS, TRAIN_SCENARIO_PROBS):
    print(f"{name:<22}: {p:.2f}")
print("No reliability-inverted examples are used for training.")
print("P13-C10 COMPLETE ✅")


In [ ]:
# ============================================================
# CELL: P13-C11 — HARDENING LOSS
# ============================================================
cfg04h = payload_04h["config"]

WATER_POS_WEIGHT = float(cfg04h["water_pos_weight"])
TYPE_TEMP_POS_WEIGHT = float(cfg04h["type_temp_pos_weight"])

LOSS_W_3CLASS = 0.50
LOSS_W_WATER = 0.75
LOSS_W_TYPE = 1.00

# New hardening-specific terms.
LOSS_W_CLEAN_ANCHOR = 0.10
LOSS_W_ROUTER = 0.15
LOSS_W_WRONG_CONF = 0.05

ANCHOR_CONFIDENCE = 0.90
WRONG_CONFIDENCE_MARGIN = 0.75

CLEAN_CODE = SCENARIO_TO_CODE["clean"]
OPTICAL_MISSING_CODE = SCENARIO_TO_CODE["optical_missing"]
SAR_MISSING_CODE = SCENARIO_TO_CODE["sar_missing"]

def hardening_loss(outputs, target, reliability, scenario_codes):
    log_probs = outputs["logits"].float()
    probs = outputs["probs"].float()
    base_prob = outputs["base_prob"].float()
    water_logit = outputs["water_logit"].float()
    type_logit = outputs["type_logit"].float()

    valid = target != IGNORE_INDEX
    water_mask = valid & ((target == 1) | (target == 2))

    # 1) Exact 3-class supervision.
    loss_3 = F.nll_loss(
        log_probs,
        target,
        ignore_index=IGNORE_INDEX,
    )

    # 2) Water-vs-background supervision.
    water_target = ((target == 1) | (target == 2)).float()
    water_pw = torch.tensor(
        WATER_POS_WEIGHT,
        device=target.device,
        dtype=torch.float32,
    )
    loss_water = F.binary_cross_entropy_with_logits(
        water_logit[valid],
        water_target[valid],
        pos_weight=water_pw,
    )

    # 3) Permanent-vs-temporary only on true water.
    type_target = (target == 2).float()
    type_pw = torch.tensor(
        TYPE_TEMP_POS_WEIGHT,
        device=target.device,
        dtype=torch.float32,
    )

    if water_mask.any():
        loss_type = F.binary_cross_entropy_with_logits(
            type_logit[water_mask],
            type_target[water_mask],
            pos_weight=type_pw,
        )
    else:
        loss_type = log_probs.sum() * 0.0

    # 4) Conservative anchor only on CLEAN samples.
    clean_samples = (scenario_codes == CLEAN_CODE)[:, None, None]

    with torch.no_grad():
        base_conf = base_prob.max(dim=1).values
        anchor_mask = valid & clean_samples & (base_conf >= ANCHOR_CONFIDENCE)

    if anchor_mask.any():
        kl_map = F.kl_div(
            log_probs,
            base_prob,
            reduction="none",
        ).sum(dim=1)
        loss_anchor = kl_map[anchor_mask].mean()
    else:
        loss_anchor = log_probs.sum() * 0.0

    # 5) Reliability-router supervision.
    # Default target: optical route follows P(clear).
    router_losses = []

    for weights in outputs["router_weights"]:
        r = F.interpolate(
            reliability,
            size=weights.shape[-2:],
            mode="bilinear",
            align_corners=False,
        )[:, 0].clamp(0.0, 1.0)

        target_opt = r.clone()

        # Modality availability overrides the soft reliability prior.
        for i in range(target_opt.shape[0]):
            code = int(scenario_codes[i].item())

            if code == OPTICAL_MISSING_CODE:
                target_opt[i].zero_()

            elif code == SAR_MISSING_CODE:
                target_opt[i].fill_(1.0)

        w_opt = weights[:, 1].float().clamp(1e-5, 1.0 - 1e-5)

        router_losses.append(
            F.binary_cross_entropy(
                w_opt,
                target_opt,
            )
        )

    loss_router = torch.stack(router_losses).mean()

    # 6) Confidence regularizer ONLY for currently wrong corrupted pixels.
    corrupted_samples = (scenario_codes != CLEAN_CODE)[:, None, None]
    pred = probs.argmax(dim=1)
    conf = probs.max(dim=1).values

    wrong_corrupt = (
        valid
        & corrupted_samples
        & (pred != target)
    )

    if wrong_corrupt.any():
        excess = (
            conf[wrong_corrupt]
            - WRONG_CONFIDENCE_MARGIN
        ).clamp_min(0.0)
        loss_wrong_conf = (excess ** 2).mean()
    else:
        loss_wrong_conf = log_probs.sum() * 0.0

    total = (
        LOSS_W_3CLASS * loss_3
        + LOSS_W_WATER * loss_water
        + LOSS_W_TYPE * loss_type
        + LOSS_W_CLEAN_ANCHOR * loss_anchor
        + LOSS_W_ROUTER * loss_router
        + LOSS_W_WRONG_CONF * loss_wrong_conf
    )

    parts = {
        "total": float(total.detach().cpu()),
        "three_class": float(loss_3.detach().cpu()),
        "water": float(loss_water.detach().cpu()),
        "type": float(loss_type.detach().cpu()),
        "clean_anchor": float(loss_anchor.detach().cpu()),
        "router": float(loss_router.detach().cpu()),
        "wrong_conf": float(loss_wrong_conf.detach().cpu()),
    }

    return total, parts

print("=" * 112)
print("P13-C11 — HARDENING LOSS")
print("=" * 112)
print("water pos weight:", WATER_POS_WEIGHT)
print("type pos weight :", TYPE_TEMP_POS_WEIGHT)
print("loss weights:", {
    "3class": LOSS_W_3CLASS,
    "water": LOSS_W_WATER,
    "type": LOSS_W_TYPE,
    "clean_anchor": LOSS_W_CLEAN_ANCHOR,
    "router": LOSS_W_ROUTER,
    "wrong_conf": LOSS_W_WRONG_CONF,
})
print("P13-C11 COMPLETE ✅")


In [ ]:
# ============================================================
# CELL: P13-C12 — ONE FIXED 3-EPOCH HARDENING RUN
#
# NO EPOCH SELECTION. NO SWEEP. FINAL EPOCH IS THE CANDIDATE.
# ============================================================
EPOCHS = 3

HEAD_LR = 1e-4
ROUTER_LR = 5e-5
RIFT_LR = 2e-5

WEIGHT_DECAY = 1e-4
MAX_GRAD_NORM = 1.0
WARMUP_FRACTION = 0.10

head_params = list(model.head.parameters())
router_params = list(model.base.routers.parameters())
rift_params = list(model.base.deep_rift.parameters())

optimizer = torch.optim.AdamW(
    [
        {"params": head_params, "lr": HEAD_LR, "name": "head"},
        {"params": router_params, "lr": ROUTER_LR, "name": "routers"},
        {"params": rift_params, "lr": RIFT_LR, "name": "deep_rift"},
    ],
    weight_decay=WEIGHT_DECAY,
)

steps_per_epoch = math.ceil(len(train_loader) / GRAD_ACCUM_STEPS)
total_steps = max(1, steps_per_epoch * EPOCHS)
warmup_steps = max(1, int(total_steps * WARMUP_FRACTION))

base_lrs = [HEAD_LR, ROUTER_LR, RIFT_LR]

def lr_lambda(step):
    if step < warmup_steps:
        return float(step + 1) / float(warmup_steps)

    progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    progress = min(max(progress, 0.0), 1.0)
    return 0.5 * (1.0 + math.cos(math.pi * progress))

scheduler = torch.optim.lr_scheduler.LambdaLR(
    optimizer,
    lr_lambda=lr_lambda,
)
scaler = torch.amp.GradScaler("cuda", enabled=AMP_ENABLED)

history = []
nonfinite_batches = 0
optimizer_steps = 0

for epoch in range(1, EPOCHS + 1):
    epoch_start = time.time()

    model.train()
    optimizer.zero_grad(set_to_none=True)

    running = {
        "total": 0.0,
        "three_class": 0.0,
        "water": 0.0,
        "type": 0.0,
        "clean_anchor": 0.0,
        "router": 0.0,
        "wrong_conf": 0.0,
    }

    scenario_counts = {name: 0 for name in TRAIN_SCENARIOS}
    counted_batches = 0

    for bi, b in enumerate(train_loader, start=1):
        sar = b["sar"].to(DEVICE, non_blocking=True)
        opt = b["optical"].to(DEVICE, non_blocking=True)
        rel = b["reliability"].to(DEVICE, non_blocking=True)
        y = b["target"].to(DEVICE, non_blocking=True)

        sar_c, opt_c, rel_c, scenario_codes, scenario_names = corrupt_training_batch(
            sar,
            opt,
            rel,
            epoch=epoch,
            batch_index=bi,
        )

        for name in scenario_names:
            scenario_counts[name] += 1

        with torch.amp.autocast("cuda", enabled=AMP_ENABLED):
            outputs = model(
                sar_c,
                opt_c,
                rel_c,
                return_aux=True,
            )

            loss, parts = hardening_loss(
                outputs,
                y,
                rel_c,
                scenario_codes,
            )

        if not torch.isfinite(loss):
            nonfinite_batches += 1
            print(f"⚠️ non-finite loss at epoch {epoch}, batch {bi}; skipped")
            optimizer.zero_grad(set_to_none=True)
            continue

        scaler.scale(loss / GRAD_ACCUM_STEPS).backward()

        for k in running:
            running[k] += parts[k]

        counted_batches += 1

        should_step = (
            bi % GRAD_ACCUM_STEPS == 0
            or bi == len(train_loader)
        )

        if should_step:
            scaler.unscale_(optimizer)

            torch.nn.utils.clip_grad_norm_(
                [p for p in model.parameters() if p.requires_grad],
                MAX_GRAD_NORM,
            )

            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            scheduler.step()

            optimizer_steps += 1

        if bi == 1 or bi % 100 == 0 or bi == len(train_loader):
            print(
                f"Epoch {epoch}/{EPOCHS} "
                f"[{bi:03d}/{len(train_loader)}] "
                f"loss={parts['total']:.4f} "
                f"head_lr={optimizer.param_groups[0]['lr']:.2e}"
            )

    means = {
        k: v / max(counted_batches, 1)
        for k, v in running.items()
    }

    row = {
        "epoch": epoch,
        "train_loss": means,
        "scenario_counts": scenario_counts,
        "optimizer_steps": optimizer_steps,
        "epoch_minutes": (time.time() - epoch_start) / 60.0,
    }
    history.append(row)

    print("=" * 112)
    print(f"EPOCH {epoch}/{EPOCHS} COMPLETE")
    print("Losses   :", {k: round(v, 5) for k, v in means.items()})
    print("Scenarios:", scenario_counts)
    print("Minutes  :", round(row["epoch_minutes"], 2))
    print("=" * 112)

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

assert nonfinite_batches == 0, "05B encountered non-finite training batches."

print("\nFixed training run complete.")
print("Optimizer steps :", optimizer_steps)
print("Nonfinite batch :", nonfinite_batches)
print("P13-C12 COMPLETE ✅")


In [ ]:
# ============================================================
# CELL: P13-C13 — FINAL CLEAN FULL-VALIDATION CHECK
# ============================================================
final_clean_full = evaluate_4way(model, val_full_loader)

print("=" * 112)
print("P13-C13 — FINAL CLEAN FULL VALIDATION")
print("=" * 112)
print("04H-CAL start:", compact_metrics(warm_metrics))
print("05B final    :", compact_metrics(final_clean_full))
print()

print("ΔmIoU      :", final_clean_full["mIoU"] - warm_metrics["mIoU"])
print("Δpermanent :", final_clean_full["permanent_water_IoU"] - warm_metrics["permanent_water_IoU"])
print("Δtemporary :", final_clean_full["temporary_flood_IoU"] - warm_metrics["temporary_flood_IoU"])

print("P13-C13 COMPLETE ✅")


In [ ]:
# ============================================================
# CELL: P13-C14 — CACHE CLEAN VAL + REFIT TEMPERATURE + RISK THRESHOLDS
# ============================================================
clean_cache = []

with torch.inference_mode():
    for bi, batch in enumerate(val_full_loader, start=1):
        sar = batch["sar"].to(DEVICE, non_blocking=True)
        opt = batch["optical"].to(DEVICE, non_blocking=True)
        rel = batch["reliability"].to(DEVICE, non_blocking=True)
        y = batch["target"].to(DEVICE, non_blocking=True)

        lp = tta_log_probs(model, sar, opt, rel)

        lp_cpu = lp.detach().cpu().half()
        y_cpu = y.detach().cpu().to(torch.int16)
        sar_cpu = sar.detach().cpu().half()
        opt_cpu = opt.detach().cpu().half()
        rel_cpu = rel.detach().cpu().half()

        scene_ids = list(batch["scene_id"])
        tile_indices = batch["tile_index"]

        for i in range(lp_cpu.shape[0]):
            clean_cache.append({
                "scene_id": str(scene_ids[i]),
                "tile_index": int(tile_indices[i]),
                "log_probs": lp_cpu[i].clone(),
                "target": y_cpu[i].clone(),
                "sar": sar_cpu[i].clone(),
                "optical": opt_cpu[i].clone(),
                "reliability": rel_cpu[i].clone(),
            })

        if bi == 1 or bi % 30 == 0 or bi == len(val_full_loader):
            print(f"[{bi:03d}/{len(val_full_loader)}] cache")

cal_entries = [x for x in clean_cache if x["scene_id"] in calibration_ids]
robust_clean_entries = [x for x in clean_cache if x["scene_id"] in robustness_ids]

assert len(cal_entries) == 34 * TILES_PER_SCENE
assert len(robust_clean_entries) == 34 * TILES_PER_SCENE

# ---- Temperature fit ----
rng = np.random.default_rng(SEED + 1305)
SAMPLE_PER_TILE = 4096

sample_lp = []
sample_y = []

for entry in cal_entries:
    lp = entry["log_probs"].float().permute(1, 2, 0).reshape(-1, NUM_CLASSES)
    y = entry["target"].reshape(-1).long()

    valid_idx = torch.nonzero(y != IGNORE_INDEX, as_tuple=False).squeeze(1)
    if valid_idx.numel() == 0:
        continue

    take = min(SAMPLE_PER_TILE, valid_idx.numel())
    chosen_np = rng.choice(valid_idx.numel(), size=take, replace=False)
    chosen = valid_idx[torch.from_numpy(chosen_np).long()]

    sample_lp.append(lp[chosen])
    sample_y.append(y[chosen])

sample_lp = torch.cat(sample_lp, dim=0).to(DEVICE)
sample_y = torch.cat(sample_y, dim=0).to(DEVICE)

log_T = nn.Parameter(torch.tensor(0.0, device=DEVICE))
opt_T = torch.optim.LBFGS(
    [log_T],
    lr=0.25,
    max_iter=60,
    tolerance_grad=1e-9,
    tolerance_change=1e-10,
    line_search_fn="strong_wolfe",
)

def temp_closure():
    opt_T.zero_grad()
    T = torch.exp(log_T).clamp(0.05, 10.0)
    scaled = F.log_softmax(sample_lp / T, dim=1)
    loss = F.nll_loss(scaled, sample_y)
    loss.backward()
    return loss

with torch.no_grad():
    nll_before_T = F.nll_loss(
        F.log_softmax(sample_lp, dim=1),
        sample_y,
    ).item()

opt_T.step(temp_closure)

TEMPERATURE_05B = float(
    torch.exp(log_T.detach()).clamp(0.05, 10.0).cpu()
)

with torch.no_grad():
    nll_after_T = F.nll_loss(
        F.log_softmax(sample_lp / TEMPERATURE_05B, dim=1),
        sample_y,
    ).item()

del sample_lp, sample_y
if torch.cuda.is_available():
    torch.cuda.empty_cache()

def probs_with_temperature(log_probs, temperature):
    return torch.softmax(log_probs.float() / float(temperature), dim=1)

def normalized_entropy(probs):
    return -(probs * probs.clamp_min(1e-8).log()).sum(dim=1) / math.log(NUM_CLASSES)

def reliability_aware_risk(probs, sar, optical, reliability):
    """
    Conservative deployment risk:
      70% predictive entropy
      30% explicit sensor/reliability risk.

    The availability terms are known at inference time, so a completely
    absent modality is never allowed to appear 'safe' solely because the
    classifier is overconfident.
    """
    ent = normalized_entropy(probs)

    sar_present = (
        sar.abs().sum(dim=1) > 1e-12
    ).float()

    opt_present = (
        optical.abs().sum(dim=1) > 1e-12
    ).float()

    pclear = reliability[:, 0].clamp(0.0, 1.0)

    sensor_risk = torch.maximum(
        1.0 - pclear,
        torch.maximum(
            1.0 - sar_present,
            1.0 - opt_present,
        ),
    )

    risk = 0.70 * ent + 0.30 * sensor_risk
    return risk.clamp(0.0, 1.0)

# ---- Freeze risk thresholds on the calibration half ----
risk_chunks = []

for entry in cal_entries:
    lp = entry["log_probs"].float().unsqueeze(0)
    y = entry["target"].long().unsqueeze(0)
    sar = entry["sar"].float().unsqueeze(0)
    opt = entry["optical"].float().unsqueeze(0)
    rel = entry["reliability"].float().unsqueeze(0)

    probs = probs_with_temperature(lp, TEMPERATURE_05B)
    risk = reliability_aware_risk(probs, sar, opt, rel)[0]

    valid = y[0] != IGNORE_INDEX
    risk_chunks.append(risk[valid].numpy().astype(np.float32))

cal_risk = np.concatenate(risk_chunks)

RISK_THRESHOLD_REVIEW_10 = float(np.quantile(cal_risk, 0.90))
RISK_THRESHOLD_REVIEW_20 = float(np.quantile(cal_risk, 0.80))

print("=" * 112)
print("P13-C14 — FINAL CONFIDENCE / RISK CALIBRATION")
print("=" * 112)
print("Temperature      :", TEMPERATURE_05B)
print("Calibration NLL before:", nll_before_T)
print("Calibration NLL after :", nll_after_T)
print("Risk threshold ~10% review:", RISK_THRESHOLD_REVIEW_10)
print("Risk threshold ~20% review:", RISK_THRESHOLD_REVIEW_20)

assert nll_after_T <= nll_before_T + 1e-7
print("P13-C14 COMPLETE ✅")


In [ ]:
# ============================================================
# CELL: P13-C15 — UNCERTAINTY / REVIEW ACCUMULATOR
# ============================================================
ECE_BINS = 15

class RobustnessAccumulator:
    def __init__(self):
        self.cm = torch.zeros((NUM_CLASSES, NUM_CLASSES), dtype=torch.long)
        self.cm_review10 = torch.zeros((NUM_CLASSES, NUM_CLASSES), dtype=torch.long)
        self.cm_review20 = torch.zeros((NUM_CLASSES, NUM_CLASSES), dtype=torch.long)

        self.n = 0
        self.sum_nll = 0.0
        self.sum_brier = 0.0
        self.sum_entropy = 0.0
        self.sum_risk = 0.0

        self.review10_n = 0
        self.review20_n = 0

        self.ece_count = np.zeros(ECE_BINS, dtype=np.int64)
        self.ece_conf_sum = np.zeros(ECE_BINS, dtype=np.float64)
        self.ece_acc_sum = np.zeros(ECE_BINS, dtype=np.float64)

        self.error_scores = []
        self.error_labels = []

    def update(self, probs, target, sar, optical, reliability):
        valid = target != IGNORE_INDEX
        if not valid.any():
            return

        probs_v = probs.permute(0, 2, 3, 1)[valid]
        target_v = target[valid].long()

        conf, pred = probs_v.max(dim=1)
        correct = pred.eq(target_v)
        err = ~correct

        ent_map = normalized_entropy(probs)
        risk_map = reliability_aware_risk(
            probs,
            sar,
            optical,
            reliability,
        )

        ent = ent_map[valid]
        risk = risk_map[valid]

        true_p = probs_v.gather(1, target_v[:, None]).squeeze(1).clamp_min(1e-8)
        nll = -true_p.log()

        onehot = F.one_hot(target_v, num_classes=NUM_CLASSES).float()
        brier = ((probs_v - onehot) ** 2).sum(dim=1)

        self.cm += confusion_from_predictions(pred, target_v).cpu()

        n = int(target_v.numel())
        self.n += n
        self.sum_nll += float(nll.sum().item())
        self.sum_brier += float(brier.sum().item())
        self.sum_entropy += float(ent.sum().item())
        self.sum_risk += float(risk.sum().item())

        bin_idx = torch.clamp((conf * ECE_BINS).long(), max=ECE_BINS - 1)

        for b in range(ECE_BINS):
            m = bin_idx == b
            if m.any():
                self.ece_count[b] += int(m.sum().item())
                self.ece_conf_sum[b] += float(conf[m].sum().item())
                self.ece_acc_sum[b] += float(correct[m].float().sum().item())

        keep10 = risk <= RISK_THRESHOLD_REVIEW_10
        keep20 = risk <= RISK_THRESHOLD_REVIEW_20

        self.review10_n += int((~keep10).sum().item())
        self.review20_n += int((~keep20).sum().item())

        if keep10.any():
            self.cm_review10 += confusion_from_predictions(
                pred[keep10],
                target_v[keep10],
            ).cpu()

        if keep20.any():
            self.cm_review20 += confusion_from_predictions(
                pred[keep20],
                target_v[keep20],
            ).cpu()

        self.error_scores.append(risk.detach().cpu().numpy().astype(np.float32))
        self.error_labels.append(err.detach().cpu().numpy().astype(np.uint8))

    def finalize(self):
        base = metrics_from_confusion(self.cm)

        ece = 0.0
        for b in range(ECE_BINS):
            if self.ece_count[b] == 0:
                continue

            avg_conf = self.ece_conf_sum[b] / self.ece_count[b]
            avg_acc = self.ece_acc_sum[b] / self.ece_count[b]

            ece += (
                self.ece_count[b] / max(self.n, 1)
            ) * abs(avg_conf - avg_acc)

        scores = np.concatenate(self.error_scores)
        labels = np.concatenate(self.error_labels)

        if np.unique(labels).size == 2:
            error_auroc = float(roc_auc_score(labels, scores))
            error_auprc = float(average_precision_score(labels, scores))
        else:
            error_auroc = float("nan")
            error_auprc = float("nan")

        rev10 = metrics_from_confusion(self.cm_review10)
        rev20 = metrics_from_confusion(self.cm_review20)

        base.update({
            "n_valid": int(self.n),
            "nll": self.sum_nll / max(self.n, 1),
            "brier": self.sum_brier / max(self.n, 1),
            "ece_15": float(ece),
            "mean_entropy": self.sum_entropy / max(self.n, 1),
            "mean_risk": self.sum_risk / max(self.n, 1),
            "risk_error_auroc": error_auroc,
            "risk_error_auprc": error_auprc,
            "review10_rate": self.review10_n / max(self.n, 1),
            "review20_rate": self.review20_n / max(self.n, 1),
            "retained10_mIoU": rev10["mIoU"],
            "retained20_mIoU": rev20["mIoU"],
        })

        return base

print("P13-C15 COMPLETE ✅")


In [ ]:
# ============================================================
# CELL: P13-C16 — UNCHANGED NOTEBOOK-05 ROBUSTNESS SUITE
# ============================================================
EVAL_SCENARIOS = [
    "clean",
    "optical_haze_proxy",
    "optical_missing",
    "sar_noise_0p35",
    "sar_missing",
    "combined_mild",
    "reliability_neutral",
    "reliability_inverted",
]

SCENARIO_SEEDS = {
    name: 51000 + i * 1000
    for i, name in enumerate(EVAL_SCENARIOS[1:], start=1)
}

def _deterministic_noise_like(tensor, seed):
    g = torch.Generator(device=tensor.device)
    g.manual_seed(int(seed))
    return torch.randn(
        tensor.shape,
        dtype=tensor.dtype,
        device=tensor.device,
        generator=g,
    )

def apply_eval_scenario(sar, optical, reliability, scenario_name, batch_index):
    sar_c = sar.clone()
    opt_c = optical.clone()
    rel_c = reliability.clone()

    if scenario_name == "clean":
        return sar_c, opt_c, rel_c

    if scenario_name == "optical_haze_proxy":
        opt_c = 0.60 * F.avg_pool2d(opt_c, kernel_size=7, stride=1, padding=3)
        rel_c = 0.25 * rel_c
        return sar_c, opt_c, rel_c

    if scenario_name == "optical_missing":
        opt_c.zero_()
        rel_c.zero_()
        return sar_c, opt_c, rel_c

    if scenario_name == "sar_noise_0p35":
        noise = _deterministic_noise_like(
            sar_c,
            SCENARIO_SEEDS[scenario_name] + int(batch_index),
        )
        sar_valid = (sar_c.abs().sum(dim=1, keepdim=True) > 0).to(sar_c.dtype)
        sar_c = sar_c + 0.35 * noise * sar_valid
        return sar_c, opt_c, rel_c

    if scenario_name == "sar_missing":
        sar_c.zero_()
        return sar_c, opt_c, rel_c

    if scenario_name == "combined_mild":
        opt_c = 0.60 * F.avg_pool2d(opt_c, kernel_size=7, stride=1, padding=3)
        rel_c = 0.25 * rel_c

        noise = _deterministic_noise_like(
            sar_c,
            SCENARIO_SEEDS[scenario_name] + int(batch_index),
        )
        sar_valid = (sar_c.abs().sum(dim=1, keepdim=True) > 0).to(sar_c.dtype)
        sar_c = sar_c + 0.25 * noise * sar_valid
        return sar_c, opt_c, rel_c

    if scenario_name == "reliability_neutral":
        rel_c.fill_(0.5)
        return sar_c, opt_c, rel_c

    if scenario_name == "reliability_inverted":
        rel_c = 1.0 - rel_c
        return sar_c, opt_c, rel_c

    raise ValueError(scenario_name)

@torch.inference_mode()
def evaluate_scenario(scenario_name):
    acc = RobustnessAccumulator()
    model.eval()

    for bi, batch in enumerate(val_robust_loader, start=1):
        sar = batch["sar"].to(DEVICE, non_blocking=True)
        opt = batch["optical"].to(DEVICE, non_blocking=True)
        rel = batch["reliability"].to(DEVICE, non_blocking=True)
        y = batch["target"].to(DEVICE, non_blocking=True)

        sar_c, opt_c, rel_c = apply_eval_scenario(
            sar,
            opt,
            rel,
            scenario_name,
            bi,
        )

        lp = tta_log_probs(
            model,
            sar_c,
            opt_c,
            rel_c,
        )

        probs = probs_with_temperature(
            lp,
            TEMPERATURE_05B,
        )

        acc.update(
            probs,
            y,
            sar_c,
            opt_c,
            rel_c,
        )

        if bi == 1 or bi % 20 == 0 or bi == len(val_robust_loader):
            print(f"[{bi:03d}/{len(val_robust_loader)}] {scenario_name}")

    return acc.finalize()

print("P13-C16 COMPLETE ✅")


In [ ]:
# ============================================================
# CELL: P13-C17 — FINAL ROBUSTNESS EVALUATION + PROMOTION DECISION
# ============================================================
BASELINE_05 = {
    "clean": {
        "mIoU": 0.8265,
        "permanent_water_IoU": 0.8885,
        "temporary_flood_IoU": 0.6204,
    },
    "optical_haze_proxy": {
        "mIoU": 0.5003,
        "permanent_water_IoU": 0.3847,
        "temporary_flood_IoU": 0.2099,
    },
    "optical_missing": {
        "mIoU": 0.2917,
        "permanent_water_IoU": 0.0148,
        "temporary_flood_IoU": 0.0000,
    },
    "sar_noise_0p35": {
        "mIoU": 0.8542,
        "permanent_water_IoU": 0.9289,
        "temporary_flood_IoU": 0.6609,
    },
    "sar_missing": {
        "mIoU": 0.5609,
        "permanent_water_IoU": 0.3026,
        "temporary_flood_IoU": 0.4288,
    },
    "combined_mild": {
        "mIoU": 0.4587,
        "permanent_water_IoU": 0.2972,
        "temporary_flood_IoU": 0.1856,
    },
    "reliability_neutral": {
        "mIoU": 0.8196,
        "permanent_water_IoU": 0.8922,
        "temporary_flood_IoU": 0.5991,
    },
    "reliability_inverted": {
        "mIoU": 0.7970,
        "permanent_water_IoU": 0.8577,
        "temporary_flood_IoU": 0.5683,
    },
}

results = []

for scenario in EVAL_SCENARIOS:
    print("\n" + "=" * 112)
    print("SCENARIO:", scenario)
    print("=" * 112)

    t0 = time.time()
    m = evaluate_scenario(scenario)

    row = {
        "scenario": scenario,
        **m,
        "minutes": (time.time() - t0) / 60.0,
    }
    results.append(row)

    print("Segmentation:", compact_metrics(m))
    print("ECE15       :", round(m["ece_15"], 6))
    print("Entropy     :", round(m["mean_entropy"], 6))
    print("Risk        :", round(m["mean_risk"], 6))
    print("Risk AUROC  :", round(m["risk_error_auroc"], 6))
    print("Review10    :", round(m["review10_rate"], 4))
    print("Review20    :", round(m["review20_rate"], 4))

results_df = pd.DataFrame(results)

for scenario in EVAL_SCENARIOS:
    base_miou = BASELINE_05[scenario]["mIoU"]
    mask = results_df["scenario"] == scenario
    results_df.loc[mask, "baseline_mIoU"] = base_miou
    results_df.loc[mask, "delta_mIoU_vs_05"] = (
        results_df.loc[mask, "mIoU"] - base_miou
    )

core_scenarios = [
    "optical_haze_proxy",
    "optical_missing",
    "sar_missing",
    "combined_mild",
]

baseline_core_avg = float(np.mean([
    BASELINE_05[s]["mIoU"]
    for s in core_scenarios
]))

new_core_avg = float(np.mean([
    float(results_df.loc[results_df["scenario"] == s, "mIoU"].iloc[0])
    for s in core_scenarios
]))

core_gain = new_core_avg - baseline_core_avg

# ----------------------------------------------------------------
# PREDECLARED PROMOTION CRITERIA
# ----------------------------------------------------------------
CLEAN_MIN_MIOU = 0.790
CLEAN_MIN_PERM = 0.810
CLEAN_MIN_TEMP = 0.570

CORE_MIN_GAIN = 0.070
OPTICAL_MISSING_MIN_MIOU = 0.380
OPTICAL_MISSING_MIN_TEMP = 0.030
SAR_MISSING_MIN_MIOU = BASELINE_05["sar_missing"]["mIoU"] - 0.030

opt_missing_row = results_df[results_df["scenario"] == "optical_missing"].iloc[0]
sar_missing_row = results_df[results_df["scenario"] == "sar_missing"].iloc[0]

criteria = {
    "clean_full_mIoU": final_clean_full["mIoU"] >= CLEAN_MIN_MIOU,
    "clean_full_permanent": final_clean_full["permanent_water_IoU"] >= CLEAN_MIN_PERM,
    "clean_full_temporary": final_clean_full["temporary_flood_IoU"] >= CLEAN_MIN_TEMP,
    "core_robustness_gain": core_gain >= CORE_MIN_GAIN,
    "optical_missing_mIoU": float(opt_missing_row["mIoU"]) >= OPTICAL_MISSING_MIN_MIOU,
    "optical_missing_temporary": float(opt_missing_row["temporary_flood_IoU"]) >= OPTICAL_MISSING_MIN_TEMP,
    "sar_missing_preserved": float(sar_missing_row["mIoU"]) >= SAR_MISSING_MIN_MIOU,
}

promoted = all(criteria.values())

print("\n" + "=" * 112)
print("P13-C17 — 05B PROMOTION DECISION")
print("=" * 112)
print("Full clean 04H-CAL:", compact_metrics(warm_metrics))
print("Full clean 05B    :", compact_metrics(final_clean_full))
print()
print("Baseline core robustness avg:", round(baseline_core_avg, 4))
print("05B core robustness avg     :", round(new_core_avg, 4))
print("Core robustness gain        :", round(core_gain, 4))
print()
for k, v in criteria.items():
    print(f"{k:<34}: {'✅' if v else '❌'}")

if promoted:
    decision = "05B_PROMOTED_RELIABILITY_HARDENED"
    print("\nDECISION: 05B PROMOTED ✅")
    print("Use 04H checkpoint + 05B delta + fixed type bias for Notebook 06.")
else:
    decision = "05B_NOT_PROMOTED_FALLBACK_04H_CAL"
    print("\nDECISION: 05B NOT PROMOTED ❌")
    print("Do NOT launch an automatic 05C.")
    print("Fallback remains the verified 04H-CAL system.")

scorecard_path = METRIC_ROOT / "FloodShift_05B_robustness_scorecard.csv"
results_df.to_csv(scorecard_path, index=False)

display(
    results_df[
        [
            "scenario",
            "baseline_mIoU",
            "mIoU",
            "delta_mIoU_vs_05",
            "permanent_water_IoU",
            "temporary_flood_IoU",
            "ece_15",
            "mean_entropy",
            "mean_risk",
            "risk_error_auroc",
            "review10_rate",
            "review20_rate",
            "retained10_mIoU",
            "retained20_mIoU",
        ]
    ]
)

print("\nSaved:", scorecard_path)
print("P13-C17 COMPLETE ✅")


In [ ]:
# ============================================================
# CELL: P13-C18 — EXPORT COMPACT DELTA + FINAL PROTOCOL
#
# We do NOT need another 409 MB transfer.
# The canonical 05B artifact is a compact delta on top of the
# already-verified 04H checkpoint.
# ============================================================
delta_prefixes = (
    "base.routers.",
    "base.deep_rift.",
    "head.",
)

delta_state = {
    k: v.detach().cpu()
    for k, v in model.state_dict().items()
    if k.startswith(delta_prefixes)
}

assert delta_state

delta_payload = {
    "experiment": "FloodShift-05B-Reliability-Hardening",
    "decision": decision,
    "promoted": bool(promoted),
    "source_04h_sha256": SHA_04H,
    "source_04e_sha256": EXPECTED_04E_SHA256,
    "type_bias": TYPE_BIAS,
    "trainable_modules": [
        "base.routers",
        "base.deep_rift",
        "head",
    ],
    "delta_state_dict": delta_state,
    "training": {
        "epochs": EPOCHS,
        "head_lr": HEAD_LR,
        "router_lr": ROUTER_LR,
        "rift_lr": RIFT_LR,
        "weight_decay": WEIGHT_DECAY,
        "grad_accum_steps": GRAD_ACCUM_STEPS,
        "effective_batch": TRAIN_BATCH_SIZE * GRAD_ACCUM_STEPS,
        "corruption_scenarios": TRAIN_SCENARIOS,
        "corruption_probabilities": TRAIN_SCENARIO_PROBS.tolist(),
        "history": history,
        "nonfinite_batches": nonfinite_batches,
    },
    "final_clean_full_validation": final_clean_full,
    "core_robustness": {
        "baseline_average": baseline_core_avg,
        "05b_average": new_core_avg,
        "gain": core_gain,
    },
    "promotion_criteria": criteria,
    "test_status": "LOCKED_NOT_EVALUATED",
}

delta_path = CHECKPOINT_ROOT / "FloodShift_05B_delta.pt"
torch.save(delta_payload, delta_path)

delta_sha = sha256_file(delta_path)

uncertainty_config = {
    "experiment": "FloodShift-05B-Uncertainty",
    "decision": decision,
    "source_04h_sha256": SHA_04H,
    "05b_delta_sha256": delta_sha,
    "type_bias": TYPE_BIAS,
    "temperature": TEMPERATURE_05B,
    "risk_score": (
        "0.70 * normalized_predictive_entropy + "
        "0.30 * max(1-Pclear, SAR_missing_indicator, optical_missing_indicator)"
    ),
    "risk_threshold_review_10": RISK_THRESHOLD_REVIEW_10,
    "risk_threshold_review_20": RISK_THRESHOLD_REVIEW_20,
    "test_status": "LOCKED_NOT_EVALUATED",
}

uncertainty_path = OUTPUT_ROOT / "FloodShift_05B_uncertainty_config.pt"
torch.save(uncertainty_config, uncertainty_path)

record = {
    "experiment": "FloodShift-05B-Reliability-Hardening",
    "decision": decision,
    "promoted": bool(promoted),
    "source_04h_sha256": SHA_04H,
    "delta_sha256": delta_sha,
    "type_bias": TYPE_BIAS,
    "warm_start_full_validation": warm_metrics,
    "final_clean_full_validation": final_clean_full,
    "baseline_05": BASELINE_05,
    "core_robustness_baseline_average": baseline_core_avg,
    "core_robustness_05b_average": new_core_avg,
    "core_robustness_gain": core_gain,
    "promotion_criteria": criteria,
    "temperature": TEMPERATURE_05B,
    "risk_threshold_review_10": RISK_THRESHOLD_REVIEW_10,
    "risk_threshold_review_20": RISK_THRESHOLD_REVIEW_20,
    "test_status": "LOCKED_NOT_EVALUATED",
}

record_path = OUTPUT_ROOT / "FloodShift_05B_decision_record.json"
with open(record_path, "w") as f:
    json.dump(record, f, indent=2)

history_path = OUTPUT_ROOT / "FloodShift_05B_history.json"
with open(history_path, "w") as f:
    json.dump(history, f, indent=2)

archive_path = OUTPUT_ROOT / "FloodShift_05B_COMPACT_ARCHIVE.zip"
with zipfile.ZipFile(archive_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for p in [
        delta_path,
        uncertainty_path,
        record_path,
        history_path,
        METRIC_ROOT / "FloodShift_05B_robustness_scorecard.csv",
    ]:
        if Path(p).exists():
            zf.write(p, arcname=Path(p).name)

print("=" * 112)
print("P13-C18 — EXPORT")
print("=" * 112)
print("Decision          :", decision)
print("05B delta         :", delta_path)
print("05B delta SHA256  :", delta_sha)
print("Delta size MB     :", round(delta_path.stat().st_size / (1024**2), 2))
print("Uncertainty config:", uncertainty_path)
print("Decision record   :", record_path)
print("Scorecard         :", METRIC_ROOT / "FloodShift_05B_robustness_scorecard.csv")
print("Compact archive   :", archive_path)
print("Frozen geo test   : LOCKED_NOT_EVALUATED")
print()
print("Download the COMPACT archive. No new ~409 MB model transfer is required.")
print("P13-C18 COMPLETE ✅")


## After the run

Send me the executed notebook or the outputs from:

- **P13-C09** — warm-start reproduction
- **P13-C12** — all 3 training epoch summaries
- **P13-C13** — final clean full-validation result
- **P13-C14** — new temperature and risk thresholds
- **P13-C17** — complete robustness scorecard + promotion decision
- **P13-C18** — exported artifact names / delta SHA

Also download:

```text
FloodShift_05B_COMPACT_ARCHIVE.zip
```

The archive contains the compact reliability-hardening delta, so we do **not** need another 409 MB checkpoint transfer.

If `05B_PROMOTED_RELIABILITY_HARDENED` is printed, Notebook 06 will reconstruct the final detector as:

\[
\text{verified 04H checkpoint}
+
\text{05B delta}
+
b_{\mathrm{type}}=-0.15
\]

and proceed to counterfactual SAR-vs-optical attribution.

If 05B is not promoted, we stop rather than launching an automatic additional hardening sweep.
